In [2]:
from __future__ import annotations

import uuid
from enum import Enum
import errno
import sys
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

from debugpy.common.log import LEVELS

In [3]:
# import my packages

from identifier import Id, CompositeId
import record_pb2
from meta import Run, Dir, File
from db import Db

In [4]:
# open database for reading

db_name = 'test-db-for-db'
path = Path(db_name)

readonly = True
# readonly = False

create=False
# create=True

the_db = Db.open(path, readonly=readonly, create=create)
env = the_db.env

In [5]:
# import identifier
# reload(identifier)
# from identifier import Id, CompositeId

In [5]:
# import db
# reload(db)
# from db import Db

In [6]:
# show lmdb env info

env.flags(), env.path(), env.info()

({'subdir': True,
  'readonly': True,
  'metasync': True,
  'sync': True,
  'map_async': False,
  'readahead': True,
  'writemap': False,
  'meminit': False,
  'lock': True},
 'test-db-for-db',
 {'map_addr': 0,
  'map_size': 4294967296,
  'last_pgno': 7,
  'last_txnid': 5,
  'max_readers': 126,
  'num_readers': 2})

In [43]:
# env.close()

In [7]:
# list all key-value pairs in the db, using lmdb's env

with env.begin(write=False) as txn:
    num_listed = 0
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            print(f'key={key}', f'value={value}')
            num_listed += 1
    print(f'num_listed={num_listed}')

key=b'd:\x00*\x00\x01' value=b'\x08\x01\x12\x04\x00*\x00\x01"\rthis\\is\\a\\dir-f\xe6@F2\x101111222233334444:\x105555666677778888'
key=b'd:\x00*\x00\x02' value=b'\x08\x01\x12\x04\x00*\x00\x02\x1a\x04\x00*\x00\x01"\x11this\\is\\a\\dir\\too-f\xe6@F2\x101111222233334444:\x109999aaaabbbbcccc'
key=b'dhd:UUffww\x88\x88:\x11\x11""33DD:\x00*\x00\x01' value=b''
key=b'dhd:\x99\x99\xaa\xaa\xbb\xbb\xcc\xcc:\x11\x11""33DD:\x00*\x00\x02' value=b''
key=b'dhf:\x11\x11""33DD:UUffww\x88\x88:\x00*\x00\x01' value=b''
key=b'dhf:\x11\x11""33DD:\x99\x99\xaa\xaa\xbb\xbb\xcc\xcc:\x00*\x00\x02' value=b''
key=b'f:\x00*\x00\x01\x00\x01' value=b'\x08\x01\x12\x06\x00*\x00\x01\x00\x01\x1a\tsome_file%\xcd\xc0\xa9E(\xb2\xf2\x192 abcd1234abcd1234abcd1234abcd1234'
key=b'f:\x00*\x00\x02\x00\x01' value=b'\x08\x01\x12\x06\x00*\x00\x02\x00\x01\x1a\x0canother_file%\xcd\xc0\xa9E(\xb2\xf2\x192 abcd1234abcd1234abcd1234abcd1234'
key=b'fh:\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124:\x00*\x00\x01\x00\x01' value=b''
key=b'

In [22]:
the_db.max_run_id()


42

In [8]:
# list runs

rec_type = b'r:'
prefix = rec_type

with env.begin(write=False) as txn, txn.cursor() as cur:
    if cur.set_range(prefix):
        while cur.key().startswith(prefix):
            run_id = int.from_bytes(cur.key()[len(rec_type):], byteorder='big')
            rec = record_pb2.RunRecord()
            rec.ParseFromString(cur.value())
            print(
                f'run {run_id},',
                f'uuid={rec.uuid},',
                f'path={rec.path},',
            )
            cur.next()

run 42, uuid=b'\xdb\xa4\x03\xf9_#JM\x80D\x1a\xfc/\xca\xbf\xd6', path=C:\some\start\dir,


In [11]:
# list directories from a specific run

rec_type = b'd:'
run_id = Id(42)
prefix = rec_type + run_id.to_bytes()

with env.begin(write=False) as txn, txn.cursor() as cur:
    if cur.set_range(prefix):
        while cur.key().startswith(prefix):
            id_parts = CompositeId.bytes_to_parts(cur.key()[2:])
            assert len(id_parts) == 2
            rec = record_pb2.DirRecord()
            rec.ParseFromString(cur.value())
            print(
                f'dir {CompositeId.bytes_to_parts(cur.key()[len(rec_type):])},',
                f'path={rec.path},',
                f'parent={CompositeId.bytes_to_parts(rec.parent_id)},',
            )
            cur.next()

dir (42, 1), path=this\is\a\dir, parent=(0,),
dir (42, 2), path=this\is\a\dir\too, parent=(42, 1),


In [13]:
# list files from a specific run

rec_type = b'f:'
run_id = Id(42)
prefix = rec_type + run_id.to_bytes()

with env.begin(write=False) as txn, txn.cursor() as cur:
    if cur.set_range(prefix):
        while cur.key().startswith(prefix):
            id_bytes = cur.key()[len(rec_type):]
            id_parts = CompositeId.bytes_to_parts(id_bytes)
            assert len(id_parts) == 3
            _, dir_id_val, file_id_val = id_parts
            rec = record_pb2.FileRecord()
            rec.ParseFromString(cur.value())
            # assert rec.id == id_bytes
            dir_key = b'd:' + id_bytes[:4]
            dir_val = txn.get(dir_key)
            if not dir_val:
                raise ValueError(f'No value found for key {dir_key}')
            dir_rec = record_pb2.DirRecord()
            dir_rec.ParseFromString(dir_val)
            print(
                f'file {CompositeId.bytes_to_parts(cur.key()[len(rec_type):])},',
                f'name={rec.name},',
                f'location={dir_rec.path},',
            )
            cur.next()

file (42, 1, 1), name=some_file, location=this\is\a\dir,
file (42, 2, 1), name=another_file, location=this\is\a\dir\too,


In [20]:
# Restore Run obj from db

rrun42 = the_db.get_run(Id(42))
rrun42.id, rrun42.root_dir


Id(42)

In [20]:
# look for duplicate file hashes

rec_type = b'fh:'
prefix = rec_type

with env.begin(write=False) as txn, txn.cursor() as cur:
    if cur.set_range(prefix):
        last_hash, last_id = b'', b''
        files_w_same_hash = []
        while cur.key().startswith(prefix):
            file_hash, file_id = cur.key().split(b':')[1:]
            if file_hash == last_hash:
                if not files_w_same_hash:
                    files_w_same_hash.append(last_id)
                files_w_same_hash.append(file_id)
            elif files_w_same_hash:
                print("Files with same hash:", [CompositeId.bytes_to_parts(f) for f in files_w_same_hash])
                files_w_same_hash = []
            last_hash, last_id = file_hash, file_id
            cur.next()

if files_w_same_hash:
    print("Files with same hash:", [CompositeId.bytes_to_parts(f) for f in files_w_same_hash])


Files with same hash: [(42, 1, 1), (42, 2, 1)]
